# COVID-19 in New York: the health–economy trade-off

Pangallo, Aleta, del Rio-Chanona, Pichler, Martín-Corral, Chinazzi, Lafond, Ajelli, Moro, Moreno,
Vespignani & Farmer (2024), *The unequal effects of the health–economy trade-off during the COVID-19
pandemic*, Nature Human Behaviour 8:264–275, in a lighter form, on the paper's own region and data:
the New York metro area from 12 February to 30 June 2020, 416,442 synthetic New Yorkers in 4,367
census tracts, and its 20 industries. People catch the virus at home, at school, at work and in
their community. Deaths make them afraid, so they stay away from work when they can and buy less
from restaurants, shops and other customer-facing industries; industries lay off the workers they no
longer need, and spending falls as unemployment rises. Pick the closures, when the measures start and
how afraid people are (the paper's Fig. 3 counterfactuals), then **Run** and watch infections and
unemployment spread tract by tract.

Seven rules (`RULES` below, from `skabm.behaviour`'s `macro`, `firm` and `person`) run on
the **JAX** engine. Where this page departs from the paper:

- **One region.** The paper models the rest of the US too; here what it buys from New York is held
  fixed, and households are one aggregate, not cohorts by age and income, so spending follows the
  region's unemployment (the paper's eq. 30). The curves show output, not value added.
- **Where people meet.** The paper builds its community and workplace contacts from Cuebiq phone
  traces it cannot share. Here people meet in their census tract, and at work the colleagues of their
  industry living in their county. Households and schools are built as the paper builds them, and
  every layer keeps the paper's contact rate (Mistry et al. 2021).
- **Calibrated values.** The paper reports its seven calibrated parameters only as posteriors (SI
  Fig. 13). The page uses the medians of the 361 accepted draws in the authors' replication package,
  and the transmissibility slider starts there. They were fitted on the paper's contacts, not these.
- **Layoffs.** The paper picks whom to lay off uniformly at random. Here each worker is drawn with the
  chance that gives the same count on average.
- **One run.** The paper averages 200 runs; the page shows one, with a fixed seed.

Data: the authors' replication package, Zenodo record 7946867 (the synthetic population, built from
the ACS and BLS; the BEA input-output tables; the essential-industry scores; the calibration draws).
Census tract centroids: the US Census Bureau's 2019 Gazetteer.

In [ ]:
# uv run --extra voila voila notebooks   (or open this notebook in Jupyter)
import io
from datetime import date, timedelta
from pathlib import Path

import ipywidgets as w
import jax
import jax.numpy as jnp
import matplotlib

matplotlib.use("Agg")  # figures become PNGs for an Image widget
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pyarrow as pa
from lonboard import Map, ScatterplotLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap

from skabm.behaviour import defaults, firm, macro, person
from skabm.behaviour.person import LATENT
from skabm.dsl import arrays, jax_tick
from skabm.player import Player, fetch

DATA = Path("data")
# One day, in the order of Methods ("Economic module", "Coupling"): the day and yesterday's
# deaths, labour demand, contacts and infections on yesterday's employment, hiring and firing,
# consumption, orders and production.
RULES = [macro.region_day, firm.industry_labour, person.place_contacts, person.exposure,
         person.compartment_progression, person.job_draws, firm.industry_production]
ZENODO = "https://zenodo.org/api/records/7946868/files/epi-econ-abm-code.zip/content"
csv = fetch(ZENODO, ["synth_pop.csv", "Z_regional.csv", "f_regional.csv", "industry_essential_score.csv",
                     "selected_pars.csv"], DATA)
GAZETTEER = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2019_Gazetteer/2019_Gaz_tracts_national.zip"
centroids = fetch(GAZETTEER, ["2019_Gaz_tracts_national.txt"], DATA)["2019_Gaz_tracts_national.txt"]

# The 20 two-digit NAICS industries in the order of the replication's tables; the
# customer-facing ones and their community contact weight, SI Table 3.
CODES = ["11", "21", "22", "23", "31-33", "42", "44-45", "48-49", "51", "52", "53", "54", "55", "56",
         "61", "62", "71", "72", "81", "92"]
FACING = {"44-45", "48-49", "61", "62", "71", "72", "81"}
COMMUNITY = {"23": 3193, "31-33": 12824, "44-45": 14091171, "48-49": 935324, "51": 1897, "52": 501401,
             "53": 27913, "54": 171610, "56": 10898, "61": 2153993, "62": 1433625, "71": 4203783,
             "72": 6006817, "81": 1646087, "92": 264724}
START = date(2020, 2, 12)  # t = 0, Methods' Timeline

# 416,442 synthetic New Yorkers; a worker was employed before the pandemic (retirees are not).
worker = (pl.col("emp_status") == "Employed") & (pl.col("occupation") != "Retired")
tract = pl.col("GEOID").cast(pl.String).str.zfill(11)
people = pl.read_csv(csv["synth_pop.csv"], null_values="NA", infer_schema_length=None).select(
    id=pl.format("p{}", "index"),
    household="hh_id",
    tract=tract,
    county=tract.str.slice(0, 5),
    age="age_group",
    worker=worker.cast(pl.Int64),
    wfh=pl.when(worker).then("work_from_home").otherwise(0),
    industry=pl.when(worker).then(pl.lit(pl.Series(CODES)).gather(pl.col("industry_id").fill_null(0))),
)
persons, places = person.contact_population(people)

# The region's input-output table (its first 20 rows and columns); what the rest of the US buys from
# it, for production or consumption, is taken as given, with the other final demand.
z = pl.read_csv(csv["Z_regional.csv"]).drop("").to_numpy()
flows = (pl.DataFrame(z[:20, :20], schema=CODES).with_columns(supplier=pl.Series(CODES))
         .unpivot(index="supplier", variable_name="buyer", value_name="amount").filter(pl.col("amount") != 0))
final = pl.read_csv(csv["f_regional.csv"]).head(20)
staff = (people.filter(pl.col("worker") == 1).group_by(id="industry")
         .agg(inperson0=(1 - pl.col("wfh")).sum().cast(pl.Float64), fromhome0=pl.col("wfh").sum().cast(pl.Float64)))
industries, orders = firm.steady_state(flows, pl.DataFrame({
    "id": CODES,
    "essential": pl.read_csv(csv["industry_essential_score.csv"])["Essential_score"],
    "customer_facing": [float(c in FACING) for c in CODES],
    "community_weight": [float(COMMUNITY.get(c, 0)) for c in CODES],
    "consumption0": final["c.ny"] + final["c.rous"],
    "government0": final["g.ny"] + final["g.rous"],
    "other0": final["oth.ny"] + final["oth.rous"] + z[:20, 20:].sum(axis=1),
}).join(staff, on="id", maintain_order="left"))
region = pl.DataFrame({"id": ["ny"], "t": [0.0], "deaths": [0.0], "reported": [0.0], "lockdown": [0.0],
                       "mandates": [0.0]})
start = arrays({"Region": region, "Industry": industries, "Order": orders, "Person": persons, "Place": places},
               RULES)

# The seven values the paper calibrates (SI §4.3) and reports only as posteriors: the medians of
# the replication package's accepted draws.
draws = pl.read_csv(csv["selected_pars.csv"])
CALIBRATED = {name: draws[column].median() for name, column in {
    "transmissibility": "beta", "fear_infection": "phi", "fear_ratio": "phiT", "fear_unemployment": "phiU",
    "reallocation_share": "deltaS", "hire_speed": "gH", "fire_speed": "gF"}.items()}
cited = defaults()
tick = jax.jit(jax_tick(RULES))
first = tick(start, {**cited, **CALIBRATED}, jax.random.key(0))  # compile now, not on the first Run
# SI footnote 2: the economy starts in its steady state, so before any death or measure
# the first day's output is the pre-pandemic one
assert np.allclose(first["Industry"]["output"], industries["output0"].to_numpy(), rtol=1e-6)

# Per tract, what the map shows: its workers out of work and its residents infected so far.
tract_ids = people["tract"].unique().sort()
where = (people["tract"].rank("dense") - 1).to_numpy()  # each person's index in tract_ids
workers = people["worker"].to_numpy().astype(float)
n_tracts = tract_ids.len()
residents = np.bincount(where, minlength=n_tracts)
jobs = np.bincount(where, weights=workers, minlength=n_tracts)
labour0 = (industries["inperson0"] + industries["fromhome0"]).sum()
output0 = industries["output0"].sum()
del people, persons, places  # Binder's 2 GB

gazetteer = (pl.scan_csv(centroids, separator="\t", infer_schema=False).rename(str.strip)
             .filter(pl.col("GEOID").is_in(tract_ids.implode())).collect())
points = pl.DataFrame({"GEOID": tract_ids}).join(gazetteer, on="GEOID", how="left", maintain_order="left").select(
    pl.col("INTPTLONG", "INTPTLAT").str.strip_chars().cast(pl.Float64))
crs = {b"ARROW:extension:name": b"geoarrow.point", b"ARROW:extension:metadata": b'{"crs": "OGC:CRS84"}'}
geometry = pa.array(points.to_numpy().tolist(), type=pa.list_(pa.float64(), 2))
table = pa.Table.from_arrays([geometry], schema=pa.schema([pa.field("geometry", geometry.type, metadata=crs)]))
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # dataviz sequential
COLOURS = ["#2a78d6", "#eb6834"]  # dataviz categorical slots 1-2
SCALE = 0.4  # the colour ramp's top, for both maps


def colour(share: np.ndarray) -> np.ndarray:
    stops = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in BLUES], dtype=float)
    at = np.linspace(0, 1, len(stops))
    rgb = np.stack([np.interp(np.clip(share, 0, 1), at, stops[:, k]) for k in range(3)], 1)
    return np.column_stack([rgb, np.full(len(share), 220)]).astype(np.uint8)


layer = ScatterplotLayer(table, get_fill_color=colour(np.zeros(n_tracts)), get_radius=12 * np.sqrt(residents),
                         radius_min_pixels=1.5, stroked=False)
city = Map(layer, view_state={"longitude": -73.95, "latitude": 40.75, "zoom": 8.6}, height=520,
           basemap=MaplibreBasemap(style=CartoStyle.Positron))


def run(params: dict, essential: np.ndarray):
    """One run, day by day: what the page draws, per tract and for the metro area."""
    state = {**start, "Industry": {**start["Industry"], "essential": jnp.asarray(essential)}}
    key = jax.random.key(0)
    for t in range(139):  # 13 February to 30 June 2020
        state = tick(state, params, jax.random.fold_in(key, t))
        person, industry = state["Person"], state["Industry"]
        yield {
            "day": START + timedelta(days=t + 1),
            "deaths": float(state["Region"]["deaths"][0]),
            "reported": float(state["Region"]["reported"][0]),
            "employment": float(jnp.sum(industry["inperson"] + industry["fromhome"]) / labour0),
            "output": float(jnp.sum(industry["output"]) / output0),
            "Out of work": 1 - np.bincount(where, weights=workers * np.asarray(person["employed"]),
                                           minlength=n_tracts) / np.maximum(jobs, 1),
            "Infected so far": np.bincount(where, weights=np.asarray(person["state"]) >= LATENT,
                                           minlength=n_tracts) / residents,
        }


def draw(history: list, k: int) -> bytes:
    steps = history[: k + 1]
    x = [s["day"] for s in steps]
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4), layout="constrained")
    left.plot(x, [s["deaths"] for s in steps], color=COLOURS[0], linewidth=2)
    left.set_title("Deaths reported per day, synthetic population", loc="left", fontsize=10)
    for i, field in enumerate(("employment", "output")):
        right.plot(x, [100 * s[field] for s in steps], color=COLOURS[i], linewidth=2, label=field)
        right.annotate(field, (x[-1], 100 * steps[-1][field]), xytext=(4, 0), textcoords="offset points",
                       color="#52514e", fontsize=9, va="center")
    right.set_title("NY metro area, % of February", loc="left", fontsize=10)
    right.legend(frameon=False, fontsize=9, loc="lower left")
    for ax in (left, right):
        ax.axvspan(START + timedelta(days=ran["start"]), START + timedelta(days=93), color="#f0efec", zorder=0)
        ax.set_xlim(START, START + timedelta(days=140))
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(color="#e6e5e0")
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        ax.tick_params(labelsize=8)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=80)
    plt.close(fig)
    return buffer.getvalue()


wide = {"description_width": "11em"}
closures = w.ToggleButtons(options=["Non-essential", "Customer-facing", "None"], description="Closures",
                           style={**wide, "button_width": "auto"},
                           tooltips=["Every non-essential activity, as New York did", "Only non-essential customer-facing activity",
                                     "No closure; schools still close and work moves home"])
begins = w.ToggleButtons(options={"17 Feb": 5, "16 Mar": 33, "30 Mar": 47}, value=33, description="Measures from",
                         style={**wide, "button_width": "8em"})
fear = w.ToggleButtons(options={"Low": 0.1, "Baseline": 1.0, "High": 10.0}, value=1.0, description="Fear of infection",
                       style={**wide, "button_width": "8em"})
beta = w.FloatSlider(CALIBRATED["transmissibility"], min=0.05, max=0.4, step=0.005, readout_format=".3f",
                     description="Transmissibility β", style=wide, layout=w.Layout(width="100%"))
shown = w.ToggleButtons(options=["Out of work", "Infected so far"], description="Map",
                        style={**wide, "button_width": "8em"})
numbers = w.HTML("Pick a scenario, then <b>Run</b>.")
picture = w.Image(format="png", layout=w.Layout(display="none"))  # until a run draws
ran = {"start": 33}
facing = industries["customer_facing"].to_numpy() == 1


def play():
    ran["start"] = begins.value
    essential = industries["essential"].to_numpy().copy()
    if closures.value != "Non-essential":
        essential[~facing if closures.value == "Customer-facing" else slice(None)] = 1.0
    params = {**cited, **CALIBRATED, "transmissibility": beta.value, "lockdown_start": float(begins.value),
              "fear_infection": CALIBRATED["fear_infection"] * fear.value}
    return run(params, essential)


def show(history: list, k: int) -> None:
    step = history[k]
    layer.get_fill_color = colour(step[shown.value] / SCALE)
    numbers.value = (f"<b>{step['day']:%d %B %Y}</b> &nbsp;·&nbsp; deaths reported so far <b>{step['reported']:.0f}</b>"
                     f" &nbsp;·&nbsp; employment <b>{100 * step['employment']:.1f}%</b>, output "
                     f"<b>{100 * step['output']:.1f}%</b> of February")
    picture.value = draw(history, k)
    picture.layout.display = None


player = Player(play, show, pace=0.1, unit="day")
shown.observe(lambda _: player.history and show(player.history, player.slider.value), names="value")
legend = w.HTML(f"<small>Each dot is a census tract, sized by its people; darker is more, up to "
                f"{100 * SCALE:.0f}%. Census Bureau 2019 Gazetteer centroids.</small>")
w.VBox([
    w.HBox([w.VBox([closures, begins, fear, beta, player.pace, player.button], layout=w.Layout(width="440px")),
            w.VBox([shown, numbers, player.slider, legend], layout=w.Layout(flex="1"))]),
    city, picture, player.errors,
])